# Timed UP and Go Template

## Import block

In [ ]:
"""
TUG Phase Segmentation (MediaPipe Pose Landmarker LITE)

Pipeline overview:
1) Extract pose landmarks per frame.
2) Build smoothed kinematic signals (posture, velocities, oscillation).
3) Segment phases using rule-based thresholds and persistence.
4) Export annotated video + per-phase timings + debug signals.

Outputs:
- annotated video: <name>_annotated.mp4
- excel: <name>_phase_times.xlsx
- debug csv: <name>_signals_debug.csv
"""

import os # standard library, no pip install needed
import math 
import urllib.request # standard library, no pip install needed
from dataclasses import dataclass # standard library, no pip install needed

import cv2 # pip install opencv-python, version = 4.13.0.92
import numpy as np # pip install numpy, version = 1.26.4
import pandas as pd # pip install pandas, version = 2.3.3
from scipy.signal import savgol_filter # pip install scipy, version = 1.15.3

import mediapipe as mp # pip install mediapipe, version = 0.10.35
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import tkinter as tk # standard library, no pip install needed
from tkinter import filedialog

## select_video_file(): a method that prompts the user to select a video and then returns the video path

In [ ]:
# ==========================================
# VIDEO SELECTION
# ==========================================
def select_video_file():
    root = tk.Tk()
    root.withdraw()
    video_path = filedialog.askopenfilename(
        title="Select TUG Video",
        filetypes=[("Video files", "*.mp4 *.avi *.mov *.mkv")]
    )
    if not video_path:
        raise RuntimeError("No video selected.")
    return video_path

## Definining parameters for the pose landmark model
The phases are being seated at the start, standing up, walking forward, turning, walking backwards, sitting down, and then being seated at the end.

### What is Savitsky-Golay?
A Savitzky–Golay filter is a digital filter that can be applied to a set of digital data points for the purpose of smoothing the data, that is, to increase the precision of the data without distorting the signal tendency.

In [4]:
# ==========================================
# CONFIG
# ==========================================
MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "pose_landmarker/pose_landmarker_lite/float16/latest/"
    "pose_landmarker_lite.task"
)
MODEL_PATH = "pose_landmarker_lite.task"

PHASES_ORDER = [
    "SITTING_START",
    "STAND_UP",
    "WALK_FORWARD",
    "TURN",
    "WALK_BACK",
    "SIT_DOWN",
    "SITTING_END",
]

SCALE_FACTOR = 1.75  # inference scale-up for higher landmark precision

SMOOTH_WIN_BASE = 15  # base Savitzky-Golay window (frames)
SMOOTH_POLY = 3  # Savitzky-Golay polynomial order

PERSIST_BASE = {  # minimum consecutive frames to confirm a transition
    "SITTING_TO_STAND": 10,
    "STAND_TO_WALK": 6,
    "WALK_TO_TURN": 6,
    "TURN_TO_WALKBACK": 6,
    "WALKBACK_TO_SITDOWN": 10,
    "SITDOWN_TO_SITTING_END": 10,
}
PERSIST_MIN_FRAMES = 3  # floor for any adaptive persistence

OSC_WIN_SEC = 0.6  # window for gait oscillation estimation
TRANSITION_BANNER_SEC = 0.45  # on-screen phase-change banner duration
MIN_WALKBACK_SEC = 1.2  # enforce minimum walk-back duration
MIN_WALK_FORWARD_SEC = 2.0  # enforce minimum walk-forward duration
MIN_TURN_SEC = 1.2  # enforce minimum turn duration
MIN_SITDOWN_SEC = 0.45  # enforce minimum sit-down duration

QUALITY_HOLD = True  # if True, gate decisions by landmark quality
QUALITY_THRESH = 0.30  # strict quality threshold
QUALITY_SOFT_MULT = 0.50  # soft threshold multiplier
QUALITY_WIN_SEC = 0.25  # rolling window for quality smoothing

# Turn fusion thresholds
TURN_OSC_DROP_FACTOR = 0.75
TURN_LATERAL_SPIKE_PCTL = 80

TURN_SEARCH_BACK_SEC = 3.0
TURN_SEARCH_FWD_SEC = 4.0
TURN_EARLY_MIN_GAP_SEC = 0.5

# Fast-walk adaptation
CADENCE_HZ_FAST = 2.4
CADENCE_HZ_VERY_FAST = 3.0

## Some helper functions

In [8]:
# ==========================================
# HELPERS
# ==========================================
"""Download the pose model if missing, return local path."""
def ensure_model(model_path: str = MODEL_PATH, url: str = MODEL_URL):
    if not os.path.exists(model_path):
        print(f"Downloading LITE model to: {model_path}")
        urllib.request.urlretrieve(url, model_path)
    return model_path


"""Safely apply Savitzky-Golay smoothing with odd window bounds."""
def safe_savgol(x, win, poly=SMOOTH_POLY):
    x = np.asarray(x, dtype=float)
    if len(x) < max(7, win):
        return x.copy()
    w = win if win % 2 == 1 else win + 1
    if w >= len(x):
        w = len(x) - 1 if (len(x) - 1) % 2 == 1 else len(x) - 2
    w = max(w, 7)
    return savgol_filter(x, window_length=w, polyorder=min(poly, w - 2))


"""Return angle ABC (degrees) using 2D landmark coordinates."""
def angle_3pt(a, b, c):
    ax, ay = a[0], a[1]
    bx, by = b[0], b[1]
    cx, cy = c[0], c[1]
    v1 = np.array([ax - bx, ay - by], dtype=float)
    v2 = np.array([cx - bx, cy - by], dtype=float)
    n1 = np.linalg.norm(v1) + 1e-9
    n2 = np.linalg.norm(v2) + 1e-9
    cosang = float(np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0))
    return float(np.degrees(np.arccos(cosang)))


"""Draw high-contrast label on video frame."""
def draw_label(frame, text, xy=(20, 45), scale=1.0):
    cv2.putText(frame, text, xy, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 6, cv2.LINE_AA)
    cv2.putText(frame, text, xy, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), 2, cv2.LINE_AA)


"""Draw a transition banner between phases."""
def draw_transition(frame, text, y=130):
    cv2.putText(frame, text, (20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 6, cv2.LINE_AA)
    cv2.putText(frame, text, (20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2, cv2.LINE_AA)


"""Extract (x, y, z) from a landmark."""
def get_landmark_xyz(landmarks, idx):
    lm = landmarks[idx]
    return (lm.x, lm.y, lm.z)


"""Compute mean visibility across a set of landmark indices."""
# value between 0 and 1
def get_lm_quality(lm, idxs):
    vis = []
    finite = []
    for i in idxs:
        p = lm[i]
        finite.append(np.isfinite(p.x) and np.isfinite(p.y) and np.isfinite(p.z))
        vis.append(float(getattr(p, "visibility", 1.0)))
    if not all(finite):
        return 0.0
    return float(np.clip(np.mean(vis), 0.0, 1.0))


"""Return first index with 'need' consecutive True values (forward)."""
# need is an int. 
# mask is a numpy array of booleans
def sustained_first_true(mask: np.ndarray, start: int, need: int):
    consec = 0
    for i in range(start, len(mask)):
        if mask[i]:
            consec += 1
            if consec >= need:
                return i - (need - 1)
        else:
            consec = 0
    return None


"""Return last index before 'end' with 'need' consecutive True values (backward)."""
def sustained_last_true_before(mask: np.ndarray, end: int, need: int):
    consec = 0
    for i in range(end, -1, -1):
        if mask[i]:
            consec += 1
            if consec >= need:
                return i
        else:
            consec = 0
    return None


## @dataclass: a decorator for creating default methods within a class as follows:
(init=True, repr=True, eq=True, order=False, unsafe_hash=False, frozen=False,
           match_args=True, kw_only=False, slots=False, weakref_slot=False)
## PhaseInterval: a class (object) with phase, start_frame, and end_frame.
Phase: a string to tell us which phase of the 7 aforementioned phases (listed in PHASES_ORDER)

Start_frame and end_frame: on which frame the phase started and ended.

There is a to_row method that returns information about the PhaseInterval, given an FPS (frames per second) value, in dictionary form.

In [9]:
@dataclass
class PhaseInterval:
    phase: str
    start_frame: int
    end_frame: int


    """Serialize interval with time columns (seconds)."""
    def to_row(self, fps):
        return {
            "phase": self.phase,
            "start_frame": self.start_frame,
            "end_frame": self.end_frame,
            "start_time_s": self.start_frame / fps,
            "end_time_s": self.end_frame / fps,
            "duration_s": max(0.0, (self.end_frame - self.start_frame) / fps),
        }


## Review these 2: estimate_cadence_hz and adapt_persist

In [10]:
"""Estimate dominant gait cadence from oscillation signal."""
# This involves Fourier Transform stuff
def estimate_cadence_hz(walk_osc: np.ndarray, fps: float):
    n = len(walk_osc)
    if n < int(3 * fps):
        return None
    a = int(0.20 * n)
    b = int(0.80 * n)
    x = np.asarray(walk_osc[a:b], dtype=float)
    x = x - np.nanmean(x)
    if np.all(np.abs(x) < 1e-9):
        return None
    freqs = np.fft.rfftfreq(len(x), d=1.0 / fps) # Return the Discrete Fourier Transform sample frequencies
    spec = np.abs(np.fft.rfft(x))
    lo, hi = 0.8, 4.0
    m = (freqs >= lo) & (freqs <= hi)
    if not np.any(m):
        return None
    return float(freqs[m][np.argmax(spec[m])])


"""Scale persistence thresholds for fast walkers."""
def adapt_persist(persist_base: dict, cadence_hz):
    persist = dict(persist_base)
    if cadence_hz is None:
        return persist, 1.0
    if cadence_hz >= CADENCE_HZ_VERY_FAST:
        scale = 0.55
    elif cadence_hz >= CADENCE_HZ_FAST:
        scale = 0.70
    else:
        scale = 1.0
    for k in persist:
        persist[k] = max(PERSIST_MIN_FRAMES, int(round(persist[k] * scale)))
    return persist, scale

## extract_signals(video_path):
This is all one function. it takes the path to the video and 

In [ ]:
# ==========================================
# SIGNAL EXTRACTION
# ==========================================
def extract_signals(video_path: str):
    """Run pose inference and build per-frame kinematic signals."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width0 = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height0 = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    model_path = ensure_model()

    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        output_segmentation_masks=False,
        num_poses=1
    )
    landmarker = vision.PoseLandmarker.create_from_options(options)

    # Landmark indices (MediaPipe Pose)
    L_SH, R_SH = 11, 12
    L_HIP, R_HIP = 23, 24
    L_KNEE, R_KNEE = 25, 26
    L_ANK, R_ANK = 27, 28
    key_idxs = [L_SH, R_SH, L_HIP, R_HIP, L_KNEE, R_KNEE, L_ANK, R_ANK]

    rows = []
    frame_idx = 0

    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        if SCALE_FACTOR != 1.0:
            frame_inf = cv2.resize(
                frame_bgr,
                (int(frame_bgr.shape[1] * SCALE_FACTOR), int(frame_bgr.shape[0] * SCALE_FACTOR)),
                interpolation=cv2.INTER_LINEAR
            )
        else:
            frame_inf = frame_bgr

        frame_rgb = cv2.cvtColor(frame_inf, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        timestamp_ms = int((frame_idx / fps) * 1000.0)

        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks and len(result.pose_landmarks) > 0:
            lm = result.pose_landmarks[0]
            q = get_lm_quality(lm, key_idxs)

            Lsh = get_landmark_xyz(lm, L_SH)
            Rsh = get_landmark_xyz(lm, R_SH)
            Lhip = get_landmark_xyz(lm, L_HIP)
            Rhip = get_landmark_xyz(lm, R_HIP)
            Lk = get_landmark_xyz(lm, L_KNEE)
            Rk = get_landmark_xyz(lm, R_KNEE)
            La = get_landmark_xyz(lm, L_ANK)
            Ra = get_landmark_xyz(lm, R_ANK)

            # hk normalized: proxy for knee extension relative to leg length
            # - high hk_norm: more extended knee (standing)
            # - low hk_norm: more flexed knee (sitting)
            hk_L = float(np.hypot(Lhip[0] - Lk[0], Lhip[1] - Lk[1]))
            hk_R = float(np.hypot(Rhip[0] - Rk[0], Rhip[1] - Rk[1]))
            hk = 0.5 * (hk_L + hk_R)

            ha_L = float(np.hypot(Lhip[0] - La[0], Lhip[1] - La[1]))
            ha_R = float(np.hypot(Rhip[0] - Ra[0], Rhip[1] - Ra[1]))
            ha = max(1e-6, 0.5 * (ha_L + ha_R))
            hk_norm = hk / ha

            # hip_y: vertical pelvis position (higher = more standing in image coords)
            hip_y = 0.5 * (Lhip[1] + Rhip[1])
            # knee_ang: average knee joint angle (larger = more extended)
            knee_ang = 0.5 * (angle_3pt(Lhip, Lk, La) + angle_3pt(Rhip, Rk, Ra))

            # centers: mid-hip and mid-ankle positions (track body translation)
            hip_cx = 0.5 * (Lhip[0] + Rhip[0])
            hip_cy = 0.5 * (Lhip[1] + Rhip[1])
            hip_z = 0.5 * (Lhip[2] + Rhip[2])

            # ankles: keep BOTH individual + center (turn detection and gait)
            Lax, Lay = La[0], La[1]
            Rax, Ray = Ra[0], Ra[1]
            ank_cx = 0.5 * (Lax + Rax)
            ank_cy = 0.5 * (Lay + Ray)

            # ank_sep_x: lateral ankle separation (gait oscillation proxy)
            ank_sep_x = (Lax - Rax)

            # turning signals: torso yaw (z vs x) and shoulder angle (2D)
            yaw_sh = math.atan2((Lsh[2] - Rsh[2]), (Lsh[0] - Rsh[0] + 1e-9))
            yaw_hip = math.atan2((Lhip[2] - Rhip[2]), (Lhip[0] - Rhip[0] + 1e-9))
            yaw_torso = 0.5 * (yaw_sh + yaw_hip)
            sh_ang_2d = math.atan2((Lsh[1] - Rsh[1]), (Lsh[0] - Rsh[0] + 1e-9))

            # facing signals: left-right ordering of shoulders/hips (sign flips with turn)
            facing_sh = (Lsh[0] - Rsh[0])
            facing_hip = (Lhip[0] - Rhip[0])

            rows.append({
                "frame": frame_idx,
                "time_s": frame_idx / fps,
                "pose_ok": 1,
                "quality": q,

                "hk_norm": hk_norm,
                "hip_y": hip_y,
                "knee_ang": knee_ang,

                "hip_cx": hip_cx,
                "hip_cy": hip_cy,
                "hip_z": hip_z,

                "L_ank_x": Lax, "L_ank_y": Lay,
                "R_ank_x": Rax, "R_ank_y": Ray,
                "ank_cx": ank_cx,
                "ank_cy": ank_cy,

                "ank_sep_x": ank_sep_x,

                "yaw_torso": yaw_torso,
                "sh_ang_2d": sh_ang_2d,

                "facing_sh": facing_sh,
                "facing_hip": facing_hip,
            })
        else:
            rows.append({
                "frame": frame_idx,
                "time_s": frame_idx / fps,
                "pose_ok": 0,
                "quality": 0.0,

                "hk_norm": np.nan,
                "hip_y": np.nan,
                "knee_ang": np.nan,

                "hip_cx": np.nan,
                "hip_cy": np.nan,
                "hip_z": np.nan,

                "L_ank_x": np.nan, "L_ank_y": np.nan,
                "R_ank_x": np.nan, "R_ank_y": np.nan,
                "ank_cx": np.nan,
                "ank_cy": np.nan,

                "ank_sep_x": np.nan,

                "yaw_torso": np.nan,
                "sh_ang_2d": np.nan,

                "facing_sh": np.nan,
                "facing_hip": np.nan,
            })

        frame_idx += 1

    cap.release()

    df = pd.DataFrame(rows)
    df.interpolate(limit_direction="both", inplace=True)

    fps_val = fps

    # quality smoothing
    qwin = max(1, int(QUALITY_WIN_SEC * fps_val))
    df["quality_s"] = df["quality"].rolling(qwin, center=True, min_periods=1).mean().values

    win = SMOOTH_WIN_BASE
    df["hk_s"] = safe_savgol(df["hk_norm"].values, win)
    df["hip_y_s"] = safe_savgol(df["hip_y"].values, win)
    df["knee_ang_s"] = safe_savgol(df["knee_ang"].values, win)
    df["hip_z_s"] = safe_savgol(df["hip_z"].values, win)

    # ankle velocities: use ankle center and each ankle for robust turning
    # ank_v_max captures the more active foot during turning
    def vel(x, y):
        return np.sqrt(np.diff(x, prepend=x[0])**2 + np.diff(y, prepend=y[0])**2) * fps_val

    df["ank_v"] = vel(df["ank_cx"].values, df["ank_cy"].values)
    df["L_ank_v"] = vel(df["L_ank_x"].values, df["L_ank_y"].values)
    df["R_ank_v"] = vel(df["R_ank_x"].values, df["R_ank_y"].values)
    df["ank_v_max"] = np.maximum(df["L_ank_v"].values, df["R_ank_v"].values)

    df["hip_v"] = vel(df["hip_cx"].values, df["hip_cy"].values)

    # lateral ankle speed (center): highlights side-to-side during turning
    df["ank_x_v"] = np.abs(np.diff(df["ank_cx"].values, prepend=df["ank_cx"].values[0])) * fps_val

    # walk oscillation amplitude from ankle separation variability
    # higher oscillation indicates steady gait; dips suggest turning or stopping
    ank_sep_s = safe_savgol(df["ank_sep_x"].values, win)
    osc_win = max(7, int(OSC_WIN_SEC * fps_val))
    kernel = np.ones(osc_win) / osc_win
    ex = np.convolve(ank_sep_s, kernel, mode="same")
    ex2 = np.convolve(ank_sep_s * ank_sep_s, kernel, mode="same")
    df["walk_osc_raw"] = np.sqrt(np.maximum(0.0, ex2 - ex * ex))

    cadence_hz = estimate_cadence_hz(df["walk_osc_raw"].values, fps_val)

    # adaptive smoothing stronger if cadence is high
    if cadence_hz is not None and cadence_hz >= CADENCE_HZ_FAST:
        win2 = min(31, max(win, 21))
    else:
        win2 = win

    df["ank_v_s"] = safe_savgol(df["ank_v"].values, win2)
    df["ank_v_max_s"] = safe_savgol(df["ank_v_max"].values, win2)
    df["hip_v_s"] = safe_savgol(df["hip_v"].values, win2)
    df["ank_x_v_s"] = safe_savgol(df["ank_x_v"].values, win2)
    df["walk_osc"] = safe_savgol(df["walk_osc_raw"].values, win2)

    # hk derivative: posture change speed (stand-up vs sit-down)
    df["hk_dot"] = np.diff(df["hk_s"].values, prepend=df["hk_s"].values[0]) * fps_val
    df["hk_dot_s"] = safe_savgol(df["hk_dot"].values, win2)

    # hip_y derivative: vertical motion speed (downward during sit-down)
    df["hip_y_dot"] = np.diff(df["hip_y_s"].values, prepend=df["hip_y_s"].values[0]) * fps_val
    df["hip_y_dot_s"] = safe_savgol(df["hip_y_dot"].values, win2)

    # hip_z derivative: depth motion speed (camera axis, forward/back movement)
    df["hip_z_dot"] = np.diff(df["hip_z_s"].values, prepend=df["hip_z_s"].values[0]) * fps_val
    df["hip_z_dot_s"] = safe_savgol(df["hip_z_dot"].values, win2)

    # yaw unwrap + rate: torso rotation speed (turning)
    yaw = np.unwrap(df["yaw_torso"].values)
    df["yaw_s"] = safe_savgol(yaw, win2)
    df["yaw_rate"] = np.abs(np.diff(df["yaw_s"].values, prepend=df["yaw_s"].values[0])) * fps_val
    df["yaw_rate_s"] = safe_savgol(df["yaw_rate"].values, win2)

    # shoulder 2D unwrap + rate: shoulder rotation speed (turning)
    sh2d = np.unwrap(df["sh_ang_2d"].values)
    df["sh2d_s"] = safe_savgol(sh2d, win2)
    df["sh2d_rate"] = np.abs(np.diff(df["sh2d_s"].values, prepend=df["sh2d_s"].values[0])) * fps_val
    df["sh2d_rate_s"] = safe_savgol(df["sh2d_rate"].values, win2)

    # facing combined + rate: left-right torso facing change (sign flips on turn)
    df["facing_sh_s"] = safe_savgol(df["facing_sh"].values, win2)
    df["facing_hip_s"] = safe_savgol(df["facing_hip"].values, win2)
    df["facing_s"] = 0.6 * df["facing_sh_s"].values + 0.4 * df["facing_hip_s"].values
    df["facing_rate"] = safe_savgol(np.abs(np.diff(df["facing_s"].values, prepend=df["facing_s"].values[0])) * fps_val, win2)

    df.attrs["cadence_hz_est"] = cadence_hz
    df.attrs["smooth_win_used"] = win2
    return df, fps, width0, height0